# Comparison of feast networks
- individual Sundays of Advent
- st. Mary feasts (Annunciatio, Purificatio, )

Building blocks

In [2]:
import pycantus
import pycantus.data as data
from pycantus.filtration import Filter
import copy

In [3]:
corpus = data.load_dataset('cantuscorpus_v1.0')

Loading chants and sources...
Data loaded!


In [4]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 888010
Number of sources in CantusCorpus v1.0: 2278


In [5]:
# Clean the corpus
# Drop doxology
doxo_filter = pycantus.filtration.Filter('doxo_filter')
doxo_filter.add_value_exclude('cantus_id', '909000')
corpus.apply_filter(doxo_filter)

# Drop fragments => sources with less than 100 chants
corpus.drop_small_sources_data(min_chants=100)

In [6]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 855146
Number of sources in CantusCorpus v1.0: 510


### Advent

In [12]:
# Build filters for Sundays, e.g. "Dom. 1 Adventus"
advent_sundays_corpora = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = copy.deepcopy(corpus)
    sunday_filter = Filter(f'{feast}_filter')
    sunday_filter.add_value_include('feast', feast)
    sunday_corpus.apply_filter(sunday_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_corpora[feast] = sunday_corpus
    print(f'{feast}: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

Dom. 1 Adventus: 6447 chants, 245 sources
Dom. 2 Adventus: 4544 chants, 240 sources
Dom. 3 Adventus: 4876 chants, 251 sources
Dom. 4 Adventus: 4689 chants, 251 sources


### Networks

In [13]:
import networkx as nx
from collections import defaultdict
import itertools

In [45]:
def info(G, fast = False):
  print("{:>12s} | '{:s}'".format('Graph', G.name))

  n = G.number_of_nodes()
  m = G.number_of_edges()
  
  print("{:>12s} | {:,d} ({:,d})".format('Nodes', n, nx.number_of_isolates(G)))
  print("{:>12s} | {:,d} ({:,d})".format('Edges', m, nx.number_of_selfloops(G)))
  print("{:>12s} | {:.2f} ({:,d})".format('Degree', 2 * m / n, max([k for _, k in G.degree()])))

  if not fast:
    C = sorted(nx.connected_components(nx.MultiGraph(G)), key = len, reverse = True)

    print("{:>12s} | {:.1f}% ({:,d})".format('Components', 100 * len(C[0]) / n, len(C)))

    print("{:>12s} | {:.4f}".format('Clustering', nx.average_clustering(G if type(G) == nx.Graph else nx.Graph(G))))
    
    C = nx.community.louvain_communities(G, weight = 'weight')
    Q = nx.community.modularity(G, C, weight = 'weight')
      
    print("{:>12s} | {:.4f} ({:,d})".format('Modularity from Louvain', Q, len(C)))
  print()

In [37]:
def build_feast_network(corpus, feast):
    # Build the network
    G = nx.Graph(name=feast)
    for source in corpus.sources:
        G.add_node(source.srclink, name=source.siglum)
    source_chants = defaultdict(set)
    for chant in corpus.chants:
        source_chants[chant.srclink].add(chant.cantus_id)
    for source1, source2 in itertools.combinations(corpus.sources, 2):
        shared_chants = source_chants[source1.srclink] & source_chants[source2.srclink]
        #print(f'{source1.siglum} and {source2.siglum} share {len(shared_chants)} chants')
        if shared_chants:
            jaccard = len(shared_chants) / len(source_chants[source1.srclink] | source_chants[source2.srclink])
            G.add_edge(source1.srclink, source2.srclink, weight=jaccard)
    return G

In [38]:
advent_sundays_nets = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = advent_sundays_corpora[feast]
    G = build_feast_network(sunday_corpus, feast)
    advent_sundays_nets[feast] = G
    print(f'{feast}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    # Save the network
    nx.write_edgelist(G, f'nets/{feast}_network.edgelist')

Dom. 1 Adventus: 245 nodes, 13735 edges
Dom. 2 Adventus: 240 nodes, 13306 edges
Dom. 3 Adventus: 251 nodes, 13927 edges
Dom. 4 Adventus: 251 nodes, 14066 edges


In [46]:
for G in advent_sundays_nets.values():
    info(G)
    print('--------------------------------------')

       Graph | 'Dom. 1 Adventus'
       Nodes | 245 (3)
       Edges | 13,735 (0)
      Degree | 112.12 (227)
  Components | 97.6% (5)
  Clustering | 0.9426
Modularity from Louvain | 0.4286 (6)

--------------------------------------
       Graph | 'Dom. 2 Adventus'
       Nodes | 240 (0)
       Edges | 13,306 (0)
      Degree | 110.88 (230)
  Components | 98.8% (2)
  Clustering | 0.9752
Modularity from Louvain | 0.4121 (4)

--------------------------------------
       Graph | 'Dom. 3 Adventus'
       Nodes | 251 (1)
       Edges | 13,927 (0)
      Degree | 110.97 (233)
  Components | 97.2% (4)
  Clustering | 0.9603
Modularity from Louvain | 0.4415 (7)

--------------------------------------
       Graph | 'Dom. 4 Adventus'
       Nodes | 251 (0)
       Edges | 14,066 (0)
      Degree | 112.08 (237)
  Components | 98.4% (2)
  Clustering | 0.9652
Modularity from Louvain | 0.4952 (4)

--------------------------------------
